# Semester 2 — AI Engineering: Práctica

**Objetivo:** Construir las piezas fundamentales de un pipeline de IA: structured outputs, function calling, caching semántico y evaluación automática.

Estas prácticas corresponden a las lecciones 2.1 a 2.6. Cada ejercicio parte del problema real que la técnica resuelve antes de mostrar la solución.

**Prerequisito:** Variable de entorno `OPENAI_API_KEY` configurada.

## Setup

In [ ]:
!pip install openai --quiet

## Ejercicio 1 — El problema del texto libre: parsear con regex

**Qué practicamos:** Pedirle al LLM que devuelva datos estructurados en texto libre y parsearlos con regex. Ver cómo y cuándo falla.

**Por qué importa arquitectónicamente:** El texto libre es el default del LLM pero es un antipatrón en pipelines. El LLM puede cambiar el formato entre ejecuciones, usar sinónimos, o añadir frases de cortesía. El regex falla silenciosamente o rompe el sistema. Esta práctica hace tangible el problema que los structured outputs resuelven.

In [ ]:
import os
import re
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def extract_with_regex(text: str) -> dict:
    """Intenta extraer nombre y edad con regex. Frágil por diseño."""
    name_match = re.search(r"(?:nombre|name)[:\s]+([A-Za-záéíóúÁÉÍÓÚñÑ\s]+)", text, re.IGNORECASE)
    age_match  = re.search(r"(?:edad|age)[:\s]+(\d+)", text, re.IGNORECASE)
    return {
        "nombre": name_match.group(1).strip() if name_match else None,
        "edad":   int(age_match.group(1))     if age_match  else None,
    }

prompts = [
    "Extrae nombre y edad: 'María tiene 34 años'",
    "Extrae nombre y edad: 'El usuario se llama Carlos y tiene treinta años'",  # edad en texto
    "Extrae nombre y edad: 'Ana, 28'",  # sin etiquetas
]

for prompt in prompts:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt + ". Responde en texto plano."}],
        temperature=0
    )
    raw = r.choices[0].message.content
    parsed = extract_with_regex(raw)
    print(f"Input:   {prompt[-40:]}")
    print(f"Raw LLM: {raw.strip()[:80]}")
    print(f"Parsed:  {parsed}")
    print()

## Ejercicio 2 — Structured Output con JSON Schema

**Qué practicamos:** El mismo caso del ejercicio anterior pero usando `response_format` con JSON schema para forzar output tipado.

**Por qué importa arquitectónicamente:** Structured outputs convierte el LLM en un generador de datos tipados que el resto del sistema puede consumir sin parsing frágil. Es la interfaz entre el mundo no-determinista del LLM y el mundo determinista del sistema.

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "persona",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "nombre": {"type": "string"},
                "edad":   {"type": ["integer", "null"]}
            },
            "required": ["nombre", "edad"],
            "additionalProperties": False
        }
    }
}

inputs = [
    "María tiene 34 años",
    "El usuario se llama Carlos y tiene treinta años",
    "Ana, 28",
]

for text in inputs:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extrae el nombre y la edad del texto. Si la edad no está disponible, usa null."},
            {"role": "user",   "content": text}
        ],
        response_format=schema,
        temperature=0
    )
    result = json.loads(r.choices[0].message.content)
    print(f"Input:  '{text}'")
    print(f"Output: {result}")
    print()

## Ejercicio 3 — Function Calling: el LLM elige la herramienta

**Qué practicamos:** Definir 3 herramientas y observar cómo el LLM decide cuál invocar (o ninguna) según el contexto de la pregunta.

**Por qué importa arquitectónicamente:** Function calling es el mecanismo por el cual un LLM puede actuar sobre sistemas externos. El LLM no ejecuta la función — solo decide invocarla y con qué argumentos. Tu sistema ejecuta la función real. Entender esta separación es fundamental para diseñar agentes seguros.

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Obtiene el tiempo actual para una ciudad.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Nombre de la ciudad"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_database",
            "description": "Busca registros en la base de datos de clientes.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "limit": {"type": "integer", "default": 10}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Envía un email a un destinatario.",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string"},
                    "subject": {"type": "string"},
                    "body": {"type": "string"}
                },
                "required": ["to", "subject", "body"]
            }
        }
    }
]

queries = [
    "¿Qué tiempo hace en Madrid?",
    "Busca los clientes que compraron en enero",
    "Manda un email a ana@example.com diciéndole que su pedido está listo",
    "¿Cuánto es 2 + 2?",  # ninguna herramienta aplica
]

for query in queries:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}],
        tools=tools,
        tool_choice="auto",
        temperature=0
    )
    msg = r.choices[0].message
    if msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"Query: '{query}'")
            print(f"  → tool: {tc.function.name}")
            print(f"  → args: {tc.function.arguments}")
    else:
        print(f"Query: '{query}'")
        print(f"  → sin tool call: '{msg.content}'")
    print()

## Ejercicio 4 — Semantic Caching básico

**Qué practicamos:** Implementar un caché semántico en memoria usando embeddings para detectar queries similares y devolver respuestas cacheadas sin llamar al LLM.

**Por qué importa arquitectónicamente:** En producción, una fracción significativa de las queries son semánticamente equivalentes aunque no sean idénticas. El semantic cache reduce latencia y coste sin degradar calidad. Es una de las optimizaciones de mayor ROI en sistemas de IA.

In [ ]:
import os
import math
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ponytail: dict en memoria, sin persistencia. Sustituir por Redis + vector index si el volumen importa.
cache: dict[str, dict] = {}  # key: query original, value: {embedding, response}

SIMILARITY_THRESHOLD = 0.92
EMBED_MODEL = "text-embedding-3-small"

def embed(text: str) -> list[float]:
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b)

def cached_completion(query: str) -> tuple[str, bool]:
    """Devuelve (respuesta, cache_hit)."""
    q_emb = embed(query)

    # Buscar en caché
    for cached_query, entry in cache.items():
        sim = cosine_similarity(q_emb, entry["embedding"])
        if sim >= SIMILARITY_THRESHOLD:
            return entry["response"], True

    # Cache miss: llamar al LLM
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}],
        temperature=0
    )
    response = r.choices[0].message.content
    cache[query] = {"embedding": q_emb, "response": response}
    return response, False

queries = [
    "¿Qué es un token en IA?",
    "¿Cuánto cuesta usar GPT-4?",
    "Explícame qué es un token",           # similar a la primera
    "¿Qué significa token en el contexto de LLMs?",  # similar a la primera
    "¿Cómo se cobra el uso de GPT-4?",    # similar a la segunda
]

for q in queries:
    response, hit = cached_completion(q)
    status = "CACHE HIT " if hit else "CACHE MISS"
    print(f"[{status}] '{q}'")
    print(f"           → {response[:80]}...")
    print()

print(f"Entradas en caché: {len(cache)}")

## Ejercicio 5 — Evaluación con LLM-as-Judge

**Qué practicamos:** Usar un LLM como evaluador automático de respuestas de otro LLM. El juez recibe la pregunta, la respuesta esperada y la respuesta generada, y devuelve un score estructurado.

**Por qué importa arquitectónicamente:** Los sistemas de IA no pueden evaluarse con asserts deterministas. Necesitas un evaluador que entienda semántica. LLM-as-judge es la técnica estándar para construir pipelines de evaluación continua sin depender de humanos en cada iteración.

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def llm_judge(question: str, expected: str, actual: str) -> dict:
    """Evalúa `actual` contra `expected` para la `question` dada. Devuelve score 0-10 + justificación."""
    system = """Eres un evaluador estricto. Dado una pregunta, una respuesta esperada y una respuesta real,
evalúa si la respuesta real es correcta, completa y precisa. Responde SOLO con JSON válido."""

    prompt = f"""Pregunta: {question}
Respuesta esperada: {expected}
Respuesta real: {actual}

Evalúa y responde con este JSON exacto:
{{"score": <0-10>, "correcto": <true/false>, "justificacion": "<máx 50 palabras>"}}"""

    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    return json.loads(r.choices[0].message.content)

# Dataset de evaluación
test_cases = [
    {
        "question": "¿Cuál es la capital de España?",
        "expected": "Madrid",
        "actual": "La capital de España es Madrid."
    },
    {
        "question": "¿Cuántos planetas tiene el sistema solar?",
        "expected": "8 planetas",
        "actual": "El sistema solar tiene 9 planetas."
    },
    {
        "question": "¿Qué es un LLM?",
        "expected": "Un modelo de lenguaje grande entrenado para predecir el siguiente token.",
        "actual": "Un LLM es una inteligencia artificial que puede mantener conversaciones."
    },
]

scores = []
for tc in test_cases:
    result = llm_judge(tc["question"], tc["expected"], tc["actual"])
    scores.append(result["score"])
    print(f"Q: {tc['question']}")
    print(f"   Score: {result['score']}/10 | Correcto: {result['correcto']}")
    print(f"   Justificación: {result['justificacion']}")
    print()

avg = sum(scores) / len(scores)
print(f"Score promedio del pipeline: {avg:.1f}/10")